In [ ]:
import copy
import inspect
import json
import math
import pickle
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "training_data"

sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("torch        =", torch.__version__)
print("cuda         =", torch.cuda.is_available())


In [ ]:
# Data and target settings
DATASET_NAME = "mean_curvature_smooth"
TARGET_INDICES = [0]
USE_GLOBAL_FEATURES = True

# Filtering and preprocessing settings
MISSING_COMPLEXITY_GROUP = {
    "dataset": "20251201",
    "timepoint": "day4p5",
    "fill_value": 2.1,
}
SPHERICITY_MAX = 0.92
SPHERICAL_MARKER_DIVERSITY_MIN = 0.5
COMPLEXITY_MIN = 2.0
INTERPOLATE_TARGET_OUTLIERS = True
OUTLIER_CLIP_QUANTILES = (0.005, 0.995)

# Split and scan settings
VAL_FRAC = 0.1
SPLIT_SEED = None
FORCED_VAL_KEYS = {
    ("20251201", "day4p5_B03_144"),
}
NUM_LAYERS = [0, 1, 2, 3, 4]

# Models to analyze
ABLATION_DEPTHS = [2, 3, 4]
CLUSTER_DEPTHS = [2, 4]

# Model settings
HIDDEN_DIM = 4 * 64
DROPOUT = 0.1
NORM = "batch"
RESIDUAL = True

# Training settings
LR = 3e-4
BATCH_SIZE = 128
MAX_EPOCHS = 2000
PATIENCE = 30
NUM_WORKERS = 4
EDGE_LOSS_WEIGHT = 0.20
EDGE_LOSS_PARAMS = {
    "weighted": False,
    "alpha": 2.0,
    "normalize_by": "graph_std",
    "clip_weight": 4.0,
}

# Experiment output settings
EXPERIMENT_GROUP = "scan_model_depth_motifs"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"run_{RUN_TIMESTAMP}"
SAVE_DIR = PROJECT_ROOT / "results_experiments" / EXPERIMENT_GROUP / RUN_NAME
FIGURES_DIR = SAVE_DIR / "figures"


In [ ]:
def _safe_filename(name):
    text = str(name).strip().replace("/", "_")
    chars = [ch if (ch.isalnum() or ch in "._-") else "_" for ch in text]
    cleaned = "".join(chars).strip("._-")
    while "__" in cleaned:
        cleaned = cleaned.replace("__", "_")
    return cleaned or "figure"


def ensure_figure_dir():
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    return FIGURES_DIR


def save_mpl_figure(fig, name, *, close=False):
    path = ensure_figure_dir() / f"{_safe_filename(name)}.pdf"
    fig.savefig(path, bbox_inches="tight", transparent=True)
    print(f"Saved figure -> {path}")
    if close:
        plt.close(fig)
    return path


def save_current_mpl_figure(name, *, close=False):
    return save_mpl_figure(plt.gcf(), name, close=close)


def save_plotly_figure(fig, name, *, scale=2):
    path = ensure_figure_dir() / f"{_safe_filename(name)}.png"
    try:
        fig.write_image(str(path), scale=scale)
        print(f"Saved Plotly figure -> {path}")
    except Exception as exc:
        print(f"Could not save Plotly figure as PNG at {path}: {exc}")
    return path


In [ ]:
from src.data.io import load_graph_dataset_from_dir, select_graph_targets
from src.data.metadata import (
    attach_metadata_to_graphs,
    load_aux_metadata_for_dir,
    load_marker_names_from_dir,
    print_graph_and_metadata_fields,
)

data_dir = DATA_ROOT / DATASET_NAME
graphs = load_graph_dataset_from_dir(str(data_dir))
print(f"Loaded {len(graphs)} organoids.")
if graphs:
    print("Raw y shape:", tuple(graphs[0].y.shape))

meta = load_aux_metadata_for_dir(str(data_dir))
attached = attach_metadata_to_graphs(graphs, meta, exclude_keys=None)
print(f"Attached metadata to {attached}/{len(graphs)} graphs.")

graphs = select_graph_targets(graphs, target_indices=TARGET_INDICES, inplace=False)
if graphs:
    print("Selected y shape:", tuple(graphs[0].y.shape))

marker_names = load_marker_names_from_dir(str(data_dir))
if marker_names is None:
    n_markers = int(graphs[0].x.size(1)) if graphs else 0
    marker_names = [f"marker_{i}" for i in range(n_markers)]
print(f"Loaded {len(marker_names)} markers.")

print_graph_and_metadata_fields(graphs)


In [ ]:
from src.data.metadata import fill_missing_metadata_for_group
from src.data.filters import (
    filter_graphs_by_marker_diversity,
    filter_graphs_by_numeric_metadata,
    filter_graphs_by_sphericity,
)
from src.data.preprocessing import interpolate_target_outliers_from_neighbors

graphs = fill_missing_metadata_for_group(
    graphs,
    field="complexity",
    fill_value=MISSING_COMPLEXITY_GROUP["fill_value"],
    dataset=MISSING_COMPLEXITY_GROUP["dataset"],
    timepoint=MISSING_COMPLEXITY_GROUP["timepoint"],
)

graphs, g_spherical = filter_graphs_by_sphericity(
    graphs,
    max_sphericity=SPHERICITY_MAX,
    print_summary=True,
    return_rejected=True,
)

g_spherical = filter_graphs_by_marker_diversity(
    g_spherical,
    min_score=SPHERICAL_MARKER_DIVERSITY_MIN,
    print_summary=True,
)

graphs = filter_graphs_by_numeric_metadata(
    graphs,
    key="complexity",
    min_value=COMPLEXITY_MIN,
    allow_missing=False,
    inplace=False,
    print_summary=True,
)

graphs = graphs + g_spherical
print(f"After filtering and spherical rescue: {len(graphs)} organoids.")

if INTERPOLATE_TARGET_OUTLIERS:
    graphs, outlier_info = interpolate_target_outliers_from_neighbors(
        graphs,
        target_indices=None,
        clip_quantiles=OUTLIER_CLIP_QUANTILES,
    )
else:
    outlier_info = None


In [ ]:
from src.data.metadata import add_log_metadata_features, promote_metadata_to_graph_tensors

field_specs = [
    {
        "meta_keys": [
            "log_surface_area",
            "log_volume",
            "log_volume_over_area",
            "log_num_cells",
        ],
        "attr_name": "global_feat",
        "kind": "graph_vector",
        "dtype": torch.float32,
    },
]

if USE_GLOBAL_FEATURES:
    graphs = add_log_metadata_features(graphs, inplace=False)
    graphs = promote_metadata_to_graph_tensors(graphs, field_specs, inplace=False)
    print("Promoted metadata fields to graph tensor attributes.")
else:
    print("Global features disabled; no global_feat attribute was attached.")


In [ ]:
from src.data.splits import graph_metadata_key, train_val_split_graphs
from src.data.metadata import infer_global_dim, snapshot_graph_metadata, strip_graph_metadata
from src.data.target_transforms import AsinhStandardizeTransform, standardize_graph_global_features

g_train, g_val, split_info = train_val_split_graphs(
    graphs,
    val_frac=VAL_FRAC,
    seed=SPLIT_SEED,
    force_val_keys=FORCED_VAL_KEYS,
    key_fn=graph_metadata_key,
)
print(f"Split -> train: {len(g_train)} | val: {len(g_val)}")

val_meta_lookup = snapshot_graph_metadata(g_val)

g_train = strip_graph_metadata(g_train, inplace=False)
g_val = strip_graph_metadata(g_val, inplace=False)

target_transform = AsinhStandardizeTransform(robust=True).fit(g_train)
target_transform.transform_graphs(g_train)
target_transform.transform_graphs(g_val)

center_global, scale_global = None, None
if USE_GLOBAL_FEATURES:
    center_global, scale_global = standardize_graph_global_features(
        g_train,
        g_val,
        attr_name="global_feat",
        robust=False,
    )

global_dim = infer_global_dim(g_train)
print("global_dim =", global_dim)


In [ ]:
from src.training.loop import TrainConfig, train
from src.training.losses import *

aux_losses = [
    WeightedLossTerm(
        name="edge",
        fn=edge_loss_term,
        weight=EDGE_LOSS_WEIGHT,
        params=EDGE_LOSS_PARAMS,
    ),
]

cfg = TrainConfig(
    lr=LR,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    aux_losses=aux_losses,
)


In [ ]:
from src.models.gnn import GINCurvature, JKGINCurvature
from src.data.metadata import infer_global_dim


trained_models = []
training_logs = []
num_layers = NUM_LAYERS

for k in num_layers:
    print(f"\n=== Training model with {k} layers ===")

    n_markers = int(g_train[0].x.size(1))
    global_dim = infer_global_dim(g_train)

    model = GINCurvature(
        n_markers=n_markers,
        global_dim=global_dim,
        hidden_dim=HIDDEN_DIM,
        num_layers=k,
        dropout=DROPOUT,      
        residual=True,
        norm=NORM      
    )


    model, metrics, history = train(model, g_train, g_val, cfg)
    trained_models.append(model)
    training_logs.append({"depth": k, "metrics": metrics, "history": history})


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.inference.predict import predict_targets
from src.analysis.marker_stats import compute_markerwise_residuals
from src.data.splits import select_graphs_by_keys, graph_metadata_key
from src.plotting.mesh_plots import (
    project_predictions_to_mesh,
    project_node_quantities_to_mesh,
)
from src.plotting.depth_scan import plot_aggregate_metric_vs_depth

In [ ]:
device = cfg.device if "cfg" in globals() else (
    "cuda" if torch.cuda.is_available() else "cpu"
)

depths = np.asarray(num_layers)
mse_by_depth = []
preds_by_depth = []
truth_by_depth = None

for depth, model_d in zip(num_layers, trained_models):
    y_d, mu_d, log_var_d, X_d = predict_targets(
        g_val,
        model_d,
        device=device,
        return_log_var=True,
        target_transform=target_transform,
    )

    mse_d = float(np.mean((mu_d - y_d) ** 2))
    mse_by_depth.append(mse_d)
    preds_by_depth.append(mu_d)

    if truth_by_depth is None:
        truth_by_depth = y_d

mse_by_depth = np.asarray(mse_by_depth)

In [ ]:
def compute_aggregate_mse_depth_stats(
    y_true_all,
    y_pred_all_by_depth,
):
    """
    Computes aggregate MSE per depth plus node-level SEM.

    Baseline = always predict global mean(y_true_all).
    """
    y_true_all = np.asarray(y_true_all).reshape(-1)

    mse_depth = []
    sem_depth = []

    for y_pred in y_pred_all_by_depth:
        y_pred = np.asarray(y_pred).reshape(-1)
        sq_err = (y_pred - y_true_all) ** 2

        mse_depth.append(float(np.mean(sq_err)))
        sem_depth.append(float(np.std(sq_err, ddof=1) / np.sqrt(len(sq_err))))

    baseline_pred = np.full_like(y_true_all, np.mean(y_true_all))
    baseline_sq_err = (baseline_pred - y_true_all) ** 2

    baseline_mse = float(np.mean(baseline_sq_err))
    baseline_sem = float(
        np.std(baseline_sq_err, ddof=1) / np.sqrt(len(baseline_sq_err))
    )

    return {
        "mse": np.asarray(mse_depth),
        "sem": np.asarray(sem_depth),
        "baseline_mse": baseline_mse,
        "baseline_sem": baseline_sem,
    }

In [ ]:
stats = compute_aggregate_mse_depth_stats(
    truth_by_depth,
    preds_by_depth,
)

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(
    depths,
    stats["mse"],
    "-o",
    linewidth=2,
    markersize=4,
    label="model",
)

ax.fill_between(
    depths,
    stats["mse"] - stats["sem"],
    stats["mse"] + stats["sem"],
    alpha=0.18,
)

ax.axhline(
    stats["baseline_mse"],
    linestyle="--",
    linewidth=2,
    color="gray",
    label="mean baseline",
)

ax.fill_between(
    depths,
    stats["baseline_mse"] - stats["baseline_sem"],
    stats["baseline_mse"] + stats["baseline_sem"],
    color="gray",
    alpha=0.18,
)

ax.set_xlabel("Model depth / number of GNN layers")
ax.set_ylabel("Validation MSE")
ax.set_title("Aggregate validation MSE vs model depth")
ax.set_xticks(depths)
ax.legend()

plt.tight_layout()
save_mpl_figure(fig, "aggregate_validation_mse_vs_model_depth")
plt.show()

In [ ]:
import organograph
from organograph.plotting.meshes import plot_organoid_mesh


def extract_one_graph_values(source_graphs, graph_index, values):
    start = sum(int(g.y.shape[0]) for g in source_graphs[:graph_index])
    end = start + int(source_graphs[graph_index].y.shape[0])
    return np.asarray(values[start:end])


def get_single_graph_and_predictions(
    graphs,
    keys,
    y_true_all,
    y_pred_all_by_depth,
    *,
    key_fn,
    meta_lookup=None,
):
    """
    Resolve a single graph (via keys) and extract its true/pred arrays
    across all depths.

    Parameters
    ----------
    graphs : list[Data]
        Full graph list (e.g. g_val).
    keys : iterable
        Stable keys (expects exactly one key here).
    y_true_all : (N,) array
        Concatenated ground truth over all graphs.
    y_pred_all_by_depth : list of (N,) arrays
        Predictions for each depth (same ordering as graphs).
    key_fn : callable
        graph_metadata_key or equivalent.
    meta_lookup : dict or None

    Returns
    -------
    graph : Data
    y_true_i : (n_i,)
    y_pred_i_by_depth : list of (n_i,)
    graph_index : int
    """
    # --- select graph ---
    selected = select_graphs_by_keys(
        graphs,
        keys,
        key_fn=key_fn,
        meta_lookup=meta_lookup,
    )

    if len(selected) != 1:
        raise ValueError(f"Expected exactly 1 graph, got {len(selected)}")

    graph = selected[0]

    # --- find its index in original list ---
    target_key = next(iter(keys))
    graph_index = None

    for i, g in enumerate(graphs):
        k = key_fn(g, meta_lookup=meta_lookup)
        if k == target_key:
            graph_index = i
            break

    if graph_index is None:
        raise ValueError("Could not find graph index in original list")

    # --- compute slice bounds ---
    sizes = [int(g.y.shape[0]) for g in graphs]
    start = sum(sizes[:graph_index])
    end = start + sizes[graph_index]

    # --- slice values ---
    y_true_i = np.asarray(y_true_all[start:end])
    y_pred_i_by_depth = [
        np.asarray(y_pred_all[start:end]) for y_pred_all in y_pred_all_by_depth
    ]

    return graph, y_true_i, y_pred_i_by_depth, graph_index


def plot_depth_predictions_grid(
    graph_index,
    graphs,
    y_true_all,
    y_pred_all_by_depth,
    depths,
    *,
    meta_lookup=None,
    colorscale="RdBu_r",
    center_at_zero=True,
    view=None,
    fig_size=(1300, 850),
):
    g = graphs[graph_index]

    y_true_i = extract_one_graph_values(graphs, graph_index, y_true_all)
    y_pred_i_by_depth = [
        extract_one_graph_values(graphs, graph_index, y_pred_all)
        for y_pred_all in y_pred_all_by_depth
    ]

    # Shared color scale across all predictions and ground truth
    all_vals = np.concatenate(
        [y_true_i.ravel()] + [yp.ravel() for yp in y_pred_i_by_depth]
    )
    finite = np.isfinite(all_vals)

    if not np.any(finite):
        vmin, vmax = -1.0, 1.0
    else:
        vv = all_vals[finite]
        if center_at_zero:
            m = float(np.max(np.abs(vv)))
            m = 1.0 if m == 0 else m
            vmin, vmax = -m, m
        else:
            vmin, vmax = float(np.min(vv)), float(np.max(vv))
            if vmin == vmax:
                vmin -= 1.0
                vmax += 1.0

    fig = make_subplots(
        rows=2,
        cols=3,
        specs=[
            [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}],
            [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}],
        ],
        subplot_titles=[
            *[f"Prediction | depth={d}" for d in depths],
            "Ground truth",
        ],
        horizontal_spacing=0.02,
        vertical_spacing=0.04,
    )

    # Plot predictions
    for k, (depth, y_pred_i) in enumerate(zip(depths, y_pred_i_by_depth)):
        row = k // 3 + 1
        col = k % 3 + 1

        result = project_node_quantities_to_mesh(
            g,
            node_true=y_true_i,
            node_pred=y_pred_i,
            meta_lookup=meta_lookup,
        )

        mesh = result["mesh"]
        mesh_pred = result["mesh_pred"]

        pred_fig = plot_organoid_mesh(
            mesh,
            vertex_values=mesh_pred,
            backend="plotly",
            colorscale=colorscale,
            center_at_zero=center_at_zero,
            vmin=vmin,
            vmax=vmax,
            show_colorbar=(k == len(depths) - 1),
            view=view,
            fig_size=fig_size,
        )

        for tr in pred_fig.data:
            fig.add_trace(tr, row=row, col=col)

        scene_key = "scene" if k == 0 else f"scene{k + 1}"
        fig.update_layout({scene_key: pred_fig.layout.scene.to_plotly_json()})

    # Ground truth in final panel
    gt_k = len(depths)
    gt_row = gt_k // 3 + 1
    gt_col = gt_k % 3 + 1

    result_gt = project_node_quantities_to_mesh(
        g,
        node_true=y_true_i,
        node_pred=y_true_i,
        meta_lookup=meta_lookup,
    )

    gt_fig = plot_organoid_mesh(
        result_gt["mesh"],
        vertex_values=result_gt["mesh_true"],
        backend="plotly",
        colorscale=colorscale,
        center_at_zero=center_at_zero,
        vmin=vmin,
        vmax=vmax,
        show_colorbar=True,
        view=view,
        fig_size=fig_size,
    )

    for tr in gt_fig.data:
        fig.add_trace(tr, row=gt_row, col=gt_col)

    scene_key = "scene6"
    fig.update_layout({scene_key: gt_fig.layout.scene.to_plotly_json()})

    fig.update_layout(
        width=fig_size[0],
        height=fig_size[1],
        title=f"{getattr(g, 'organoid_str', f'graph {graph_index}')} | predictions by depth",
    )

    return fig



def plot_depth_predictions_grid_for_key(
    graphs,
    keys,
    y_true_all,
    y_pred_all_by_depth,
    depths,
    *,
    key_fn,
    meta_lookup=None,
    colorscale="RdBu_r",
    center_at_zero=True,
    view=None,
    fig_size=(1300, 850),
):
    """
    Plot predictions across depths for the single graph identified by `keys`.

    Assumes `keys` contains exactly one graph key, e.g.
    split_info["forced_val_keys"].
    """

    keys = list(keys)
    if len(keys) != 1:
        raise ValueError(f"Expected exactly one key, got {len(keys)}")

    target_key = keys[0]

    graph_index = None
    for i, g in enumerate(graphs):
        if key_fn(g, meta_lookup=meta_lookup) == target_key:
            graph_index = i
            break

    if graph_index is None:
        raise ValueError(f"Could not find graph with key={target_key!r}")

    return plot_depth_predictions_grid(
        graph_index,
        graphs,
        y_true_all,
        y_pred_all_by_depth,
        depths,
        meta_lookup=meta_lookup,
        colorscale=colorscale,
        center_at_zero=center_at_zero,
        view=view,
        fig_size=fig_size,
    )

In [ ]:
fig = plot_depth_predictions_grid_for_key(
    g_val,
    split_info["forced_val_keys"],
    truth_by_depth,
    preds_by_depth,
    depths,
    key_fn=graph_metadata_key,
    meta_lookup=val_meta_lookup,
    view=dict(azim=-135, elev=55),
)

save_plotly_figure(fig, "depth_prediction_mesh_grid")
fig.show()

In [ ]:
from src.data.subgraphs import build_ego_subgraphs_for_dataset
from src.data.subgraph_sampling import sample_subgraphs_coverage, print_sampling_summary
from src.analysis.perturbation import compute_perturbation_influence_maps
from src.plotting.influence_maps import (
    plot_influence_heatmap,
    plot_influence_center_resolved,
)

# depths to run ablation for
ablation_depths = ABLATION_DEPTHS

# assumes:
# num_layers      = list/array of trained depths
# trained_models  = list of trained models in same order
depth_to_model = dict(zip(num_layers, trained_models))

ablation_results = {}

for depth in ablation_depths:
    print(f"\n==============================")
    print(f"Ablation analysis for depth={depth}")
    print(f"==============================")

    model_d = depth_to_model[depth]

    # Build ego-subgraphs with radius matching the model depth
    val_subgraphs_d = build_ego_subgraphs_for_dataset(
        g_val,
        num_hops=depth,
        max_centers_per_graph=None,
        seed=0,
    )

    sampled_val_subgraphs_d, sample_info_d = sample_subgraphs_coverage(
        val_subgraphs_d,
        marker_names=marker_names,
        k_hops=depth,
        max_subgraphs=3000,
        min_center_count=40,
        min_pair_count=25,
        seed=0,
    )

    print_sampling_summary(sample_info_d, marker_names)

    res_d = compute_perturbation_influence_maps(
        subgraphs=sampled_val_subgraphs_d,
        model=model_d,
        marker_names=marker_names,
        k_hops=depth,
        mode="single",
        max_subgraphs=None,
        batch_size=128,
    )

    ablation_results[depth] = {
        "res": res_d,
        "sample_info": sample_info_d,
        "sampled_subgraphs": sampled_val_subgraphs_d,
    }

    # Center-marker resolved effects
    fig, axes = plot_influence_center_resolved(
        res_d["delta_mu_cmarker"],
        res_d["hops"],
        res_d["marker_names"],
        title=f"Depth={depth} | Δμ per perturbed marker (X) conditioned on center marker (Y)",
        sort_center=False,
        center_zero=True,
        cmap="RdBu_r",
    )
    save_mpl_figure(fig, f"ablation_depth_{depth}_delta_mu_center_resolved")
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

from src.analysis.motif_clustering import run_embedding_clustering
from src.analysis.marker_stats import append_none_marker_column
from src.analysis.cluster_analysis import (
    cluster_marker_means,
    cluster_marker_distribution,
    cluster_order_from_values,
    build_cluster_exemplar_subgraphs,
)
from src.plotting.cluster_plots import (
    plot_cluster_marker_heatmap,
    plot_cluster_boxplots,
)
from src.plotting.motif_plots import (
    DEFAULT_MARKER_COLORS,
    plot_cluster_exemplar_subgraphs_radial,
    plot_marker_color_legend,
    plot_cluster_exemplars_full_organoid_rows,
)

In [ ]:
cluster_depths = CLUSTER_DEPTHS
depth_to_model = dict(zip(num_layers, trained_models))

cluster_results_by_depth = {}

for depth in cluster_depths:
    print(f"\n==============================")
    print(f"Embedding clustering | depth={depth}")
    print(f"==============================")

    model_d = depth_to_model[depth]

    extraction_d, clustering_result_d, summary_d = run_embedding_clustering(
        graphs=g_val,
        model=model_d,
        batch_size=128,
        center_only=False,
        embedding_variant="local",
        residualize_global=False,
        clustering="gmm",
        n_clusters=6,
        k_values=range(2, 11),
        covariance_type="full",
        standardize=True,
        pca_dim=None,
        seed=0,
        marker_names=marker_names,
    )

    Y_true_d, Y_pred_d, log_var_d = target_transform.inverse_distribution(
        extraction_d.y_true,
        extraction_d.y_pred,
        log_var=extraction_d.log_var,
    )

    labels_d = clustering_result_d.labels

    # labels_raw_d = clustering_result_d.labels

    # # Rename clusters by increasing median Y_true:
    # # new C0 = cluster with lowest median true curvature, etc.
    # unique_clusters = np.unique(labels_raw_d)

    # cluster_medians = {
    #     c: np.nanmedian(Y_true_d[labels_raw_d == c])
    #     for c in unique_clusters
    # }

    # old_to_new = {
    #     old_c: new_c
    #     for new_c, old_c in enumerate(
    #         sorted(unique_clusters, key=lambda c: cluster_medians[c])
    #     )
    # }

    # labels_d = np.asarray([old_to_new[c] for c in labels_raw_d], dtype=int)

    # # Optional: overwrite clustering_result labels so downstream helper functions
    # # that use clustering_result also see the renamed clusters
    # clustering_result_d.labels = labels_d

    X_ext_d, marker_names_ext = append_none_marker_column(
        extraction_d.x_markers,
        marker_names,
    )

    marker_means_d = cluster_marker_means(X_ext_d, labels_d)
    dist_d = cluster_marker_distribution(X_ext_d, labels_d)

    cluster_order_d = cluster_order_from_values(
        labels_d,
        Y_pred_d,
        reducer=np.mean,
    )

    cluster_exemplars_d = build_cluster_exemplar_subgraphs(
        graphs=g_val,
        extraction=extraction_d,
        clustering_result=clustering_result_d,
        top_k_per_cluster=4,
        num_hops=depth,
        require_assigned_label=True,
    )

    cluster_results_by_depth[depth] = {
        "model": model_d,
        "extraction": extraction_d,
        "clustering_result": clustering_result_d,
        "summary": summary_d,
        "Y_true": Y_true_d,
        "Y_pred": Y_pred_d,
        "log_var": log_var_d,
        "labels": labels_d,
        "X_ext": X_ext_d,
        "marker_names_ext": marker_names_ext,
        "marker_means": marker_means_d,
        "marker_distribution": dist_d,
        "cluster_order": cluster_order_d,
        "cluster_exemplars": cluster_exemplars_d,
    }

    print(f"depth={depth}: n_clusters={len(np.unique(labels_d))}")

In [ ]:
fig, axes = plt.subplots(
    1,
    len(cluster_depths),
    figsize=(5.5 * len(cluster_depths), 4.5),
    squeeze=False,
)

axes = axes.ravel()

for ax, depth in zip(axes, cluster_depths):
    out = cluster_results_by_depth[depth]

    plot_cluster_marker_heatmap(
        out["marker_means"],
        out["marker_names_ext"],
        title=f"Depth={depth} | marker means | K={len(np.unique(out['labels']))}",
        ax=ax,
        show_colorbar=(depth == cluster_depths[-1]),
    )

plt.tight_layout()
save_mpl_figure(fig, "cluster_marker_means_by_depth")
plt.show()

In [ ]:
fig, axes = plt.subplots(
    1,
    len(cluster_depths),
    figsize=(5.5 * len(cluster_depths), 4.5),
    squeeze=False,
)

axes = axes.ravel()

for ax, depth in zip(axes, cluster_depths):
    out = cluster_results_by_depth[depth]

    plot_cluster_marker_heatmap(
        out["marker_distribution"],
        out["marker_names_ext"],
        title=f"Depth={depth} | marker distribution | K={len(np.unique(out['labels']))}",
        ax=ax,
        show_colorbar=(depth == cluster_depths[-1]),
        colorbar_label="fraction of marker-positive nodes in cluster",
    )

plt.tight_layout()
save_mpl_figure(fig, "cluster_marker_distribution_by_depth")
plt.show()

In [ ]:
fig, axes = plt.subplots(
    1,
    len(cluster_depths),
    figsize=(6.0 * len(cluster_depths), 4.5),
    squeeze=False,
)

axes = axes.ravel()

for ax, depth in zip(axes, cluster_depths):
    out = cluster_results_by_depth[depth]

    plot_cluster_boxplots(
        [out["Y_true"], out["Y_pred"]],
        out["labels"],
        data_labels=["true", "predicted"],
        colors=["lightblue", "lightgreen"],
        ylabel="curvature",
        title=f"Depth={depth} | true vs predicted | K={len(np.unique(out['labels']))}",
        cluster_order=out["cluster_order"],
        ax=ax,
    )

plt.tight_layout()
save_mpl_figure(fig, "cluster_true_vs_predicted_boxplots_by_depth")
plt.show()

In [ ]:
fig, ax = plot_marker_color_legend(marker_colors=DEFAULT_MARKER_COLORS)
save_mpl_figure(fig, "marker_color_legend")
plt.show()

for depth in cluster_depths:
    out = cluster_results_by_depth[depth]
    cluster_exemplars_d = out["cluster_exemplars"]

    print("\n==============================")
    print(f"Radial cluster exemplars | depth={depth}")
    print(f"==============================")

    for cluster_id in sorted(cluster_exemplars_d):
        fig_axes = plot_cluster_exemplar_subgraphs_radial(
            cluster_exemplars_d,
            cluster_id=cluster_id,
            marker_names=marker_names,
            marker_colors=DEFAULT_MARKER_COLORS,
        )
        if fig_axes is not None:
            fig, axes = fig_axes
            save_mpl_figure(fig, f"radial_cluster_exemplars_depth_{depth}_cluster_{cluster_id}")
        plt.show()

In [ ]:
from plotly.subplots import make_subplots

from src.plotting.mesh_plots import (
    project_marker_categories_for_graph,
    project_cluster_categories_for_graph,
    plot_categorical_mesh,
    cluster_color_map,
)

In [ ]:
# assumes:
# g_val_selected contains exactly the selected graph
# cluster_results_by_depth[depth]["labels"] contains labels for all nodes in g_val
# cluster_depths = CLUSTER_DEPTHS


g_val_selected = select_graphs_by_keys(
    g_val,
    split_info["forced_val_keys"],
    key_fn=graph_metadata_key,
    meta_lookup=val_meta_lookup,
)

selected_meta_lookup = {
    getattr(g, "organoid_str"): val_meta_lookup[getattr(g, "organoid_str")]
    for g in g_val_selected
}

g = g_val_selected[0]

# Find selected graph index in g_val by organoid_str
selected_key = getattr(g, "organoid_str")
graph_index = [
    i for i, gg in enumerate(g_val)
    if getattr(gg, "organoid_str") == selected_key
][0]

start = sum(int(gg.y.shape[0]) for gg in g_val[:graph_index])
end = start + int(g.y.shape[0])

figs = []
titles = []

# --- Fate marker mesh ---
marker_proj = project_marker_categories_for_graph(
    g,
    marker_names=marker_names,
    marker_colors=DEFAULT_MARKER_COLORS,
    meta_lookup=val_meta_lookup,
    missing_category="none",
)

fig_marker = plot_categorical_mesh(
    marker_proj["mesh"],
    marker_proj["mesh_categories"],
    category_order=[
        m for m in DEFAULT_MARKER_COLORS
        if m != "none" and m in marker_names
    ],
    category_colors=DEFAULT_MARKER_COLORS,
    fig_size=(700, 600),
    view=dict(azim=-135, elev=50),
    baseline_color=DEFAULT_MARKER_COLORS["none"],
    baseline_name="none",
    add_legend=True,
)

figs.append(fig_marker)
titles.append("Fate markers")

# --- Cluster meshes for each depth ---
for depth in cluster_depths:
    labels_all_d = cluster_results_by_depth[depth]["labels"]
    labels_i_d = np.asarray(labels_all_d[start:end], dtype=int)

    cluster_proj = project_cluster_categories_for_graph(
        g,
        labels_i_d,
        meta_lookup=val_meta_lookup,
        missing_category=None,
    )

    K = int(np.max(labels_i_d)) + 1
    cluster_colors_raw = cluster_color_map(K, cmap_name="tab10")
    cluster_colors = {f"C{k}": cluster_colors_raw[k] for k in range(K)}

    fig_cluster = plot_categorical_mesh(
        cluster_proj["mesh"],
        cluster_proj["mesh_categories"],
        category_order=[f"C{k}" for k in range(K)],
        category_colors=cluster_colors,
        fig_size=(700, 600),
        view=dict(azim=-135, elev=50),
        baseline_color="lightgray",
        baseline_name=None,
        add_legend=True,
    )

    figs.append(fig_cluster)
    titles.append(f"Clusters | depth={depth} | K={K}")

In [ ]:
n_cols = len(figs)

fig = make_subplots(
    rows=1,
    cols=n_cols,
    specs=[[{"type": "scene"} for _ in range(n_cols)]],
    subplot_titles=titles,
    horizontal_spacing=0.02,
)

for col, subfig in enumerate(figs, start=1):
    for tr in subfig.data:
        fig.add_trace(tr, row=1, col=col)

    scene_name = "scene" if col == 1 else f"scene{col}"
    if subfig.layout.scene is not None:
        fig.layout[scene_name].update(
            subfig.layout.scene.to_plotly_json()
        )

fig.update_layout(
    width=450 * n_cols,
    height=550,
    title=f"{selected_key} | fate markers and cluster maps by model depth",
    legend=dict(x=1.02, y=1.0),
)

save_plotly_figure(fig, "marker_and_cluster_meshes_by_depth")
fig.show()

In [ ]:
def _jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, torch.dtype):
        return str(obj)
    if isinstance(obj, np.dtype):
        return str(obj)
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def save_joint_experiment(save_dir, config, results_by_experiment, notes=None):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    with open(save_dir / "config.json", "w") as f:
        json.dump(_jsonable(config), f, indent=2)

    with open(save_dir / "results.pkl", "wb") as f:
        pickle.dump(results_by_experiment, f)

    meta = {
        "timestamp": datetime.now().isoformat(),
        "notes": notes,
        "experiments": list(results_by_experiment.keys()),
    }
    with open(save_dir / "meta.json", "w") as f:
        json.dump(_jsonable(meta), f, indent=2)

    print(f"Saved joint experiment -> {save_dir}")


In [ ]:
scan_results_compact = {
    "depths": depths,
    "mse_by_depth": mse_by_depth,
    "stats": stats,
    "truth_by_depth": truth_by_depth,
    "preds_by_depth": preds_by_depth,
}

ablation_results_compact = {
    depth: {
        "result": pack["res"],
        "sample_info": pack["sample_info"],
    }
    for depth, pack in ablation_results.items()
}

cluster_results_compact = {
    depth: {
        "summary": pack["summary"],
        "Y_true": pack["Y_true"],
        "Y_pred": pack["Y_pred"],
        "log_var": pack["log_var"],
        "labels": pack["labels"],
        "marker_names_ext": pack["marker_names_ext"],
        "marker_means": pack["marker_means"],
        "marker_distribution": pack["marker_distribution"],
        "cluster_order": pack["cluster_order"],
    }
    for depth, pack in cluster_results_by_depth.items()
}

joint_config = {
    "experiment_group": EXPERIMENT_GROUP,
    "dataset_name": DATASET_NAME,
    "target_indices": TARGET_INDICES,
    "use_global_features": USE_GLOBAL_FEATURES,
    "filters": {
        "missing_complexity_group": MISSING_COMPLEXITY_GROUP,
        "sphericity_max": SPHERICITY_MAX,
        "spherical_marker_diversity_min": SPHERICAL_MARKER_DIVERSITY_MIN,
        "complexity_min": COMPLEXITY_MIN,
        "interpolate_target_outliers": INTERPOLATE_TARGET_OUTLIERS,
        "outlier_clip_quantiles": OUTLIER_CLIP_QUANTILES,
    },
    "split": {
        "val_frac": VAL_FRAC,
        "split_seed": SPLIT_SEED,
        "forced_val_keys": FORCED_VAL_KEYS,
    },
    "num_layers": NUM_LAYERS,
    "ablation_depths": ABLATION_DEPTHS,
    "cluster_depths": CLUSTER_DEPTHS,
    "hidden_dim": HIDDEN_DIM,
    "dropout": DROPOUT,
    "norm": NORM,
    "residual": RESIDUAL,
    "train_config": {
        "lr": LR,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "num_workers": NUM_WORKERS,
        "edge_loss_weight": EDGE_LOSS_WEIGHT,
        "edge_loss_params": EDGE_LOSS_PARAMS,
    },
    "field_specs": field_specs,
    "n_train_graphs": len(g_train),
    "n_val_graphs": len(g_val),
    "marker_names": list(marker_names),
    "figures_dir": str(FIGURES_DIR),
}

results_by_experiment = {
    "depth_scan": scan_results_compact,
    "ablation_results": ablation_results_compact,
    "cluster_results": cluster_results_compact,
    "training_logs": training_logs,
}

save_joint_experiment(
    save_dir=SAVE_DIR,
    config=joint_config,
    results_by_experiment=results_by_experiment,
    notes="Depth scan, perturbation maps, and motif clustering analysis.",
)
